In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

In [ ]:
amst_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_final_beta.csv")

In [ ]:
lnd_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_final_beta.csv")

In [ ]:
def extract_param(beta_df, param_name):
    row = beta_df.loc[beta_df["Name"] == param_name]
    if row.empty:
        return np.nan, np.nan
    return row["Value"].values[0], row["Robust std err."].values[0]


def wald_test(amst_beta, lnd_beta, param_name, alpha=0.05):
    
    b_ams, se_ams = extract_param(amst_beta, param_name)
    b_lnd, se_lnd = extract_param(lnd_beta, param_name)

    # Wald statistic: difference / pooled SE (independent samples)
    pooled_se = np.sqrt(se_ams**2 + se_lnd**2)
    w_stat = (b_ams - b_lnd) / pooled_se

    # Two-sided p-value from standard normal
    p_value = 2 * norm.sf(np.abs(w_stat))
    reject = p_value < alpha

    return {
        "parameter": param_name,
        "AMS_coeff": round(b_ams, 4),
        "AMS_se": round(se_ams, 4),
        "LDN_coeff": round(b_lnd, 4),
        "LDN_se": round(se_lnd, 4),
        "difference": round(b_ams - b_lnd, 4),
        "pooled_se": round(pooled_se, 4),
        "W_stat": round(w_stat, 4),
        "p_value": round(p_value, 6),
        "reject_H0": reject,
    }

In [ ]:
h1_time_pt   = wald_test(amst_final_beta, lnd_final_beta, "b_time_pt")
h1_time_bike = wald_test(amst_final_beta, lnd_final_beta, "b_time_bike")

h2_income_pt   = wald_test(amst_final_beta, lnd_final_beta, "b_income_pt")
h2_income_bike = wald_test(amst_final_beta, lnd_final_beta, "b_income_bike")

exp_age_pt = wald_test(amst_final_beta, lnd_final_beta, "b_age_pt")
exp_age_bike = wald_test(amst_final_beta, lnd_final_beta, "b_age_bike")
exp_dist_bike = wald_test(amst_final_beta, lnd_final_beta, "b_dist_bike")

In [ ]:
all_tests = [
    h1_time_pt, h1_time_bike,       # H1
    h2_income_pt, h2_income_bike,    # H2
    exp_age_pt, exp_age_bike,        # Exploratory
    exp_dist_bike,                   # Exploratory
]

wald_summary = pd.DataFrame(all_tests)

wald_summary["hypothesis"] = [
    "H1", "H1", "H2", "H2", "Exploratory", "Exploratory", "Exploratory"
]

col_order = [
    "hypothesis", "parameter",
    "AMS_coeff", "AMS_se", "LDN_coeff", "LDN_se",
    "difference", "W_stat", "p_value", "reject_H0",
]
wald_summary = wald_summary[col_order]

In [ ]:
print("WALD TEST SUMMARY — All Cross-City Comparisons")
print("=" * 100)
print(wald_summary.to_string(index=False))